# Information Bottleneck Tutorial

## Introduction

The **Information Bottleneck (IB)** is a fundamental concept in information theory that provides a principled approach to compress information while preserving what's relevant for a specific task. Originally introduced by Tishby, Pereira, and Bialek in 1999, the IB method has found applications in machine learning, deep learning theory, clustering, and feature extraction.

### Core Idea

Given two random variables:
- **X**: The input or observation
- **Y**: The target or relevant variable (not directly observable)

We want to find a compressed representation **T** of **X** that:
1. Minimizes the information about X (compression)
2. Maximizes the information about Y (preserves relevant information)

This creates an "information bottleneck" where we squeeze through only the information that's relevant for predicting Y.

In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.special import rel_entr
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

## 1. Mathematical Foundations

### 1.1 Information Theory Basics

Before diving into the Information Bottleneck, let's review key concepts:

#### Entropy
The entropy of a random variable X measures the average uncertainty:

$$H(X) = -\sum_{x} p(x) \log p(x)$$

#### Mutual Information
The mutual information between X and Y measures how much knowing one variable reduces uncertainty about the other:

$$I(X; Y) = \sum_{x,y} p(x,y) \log \frac{p(x,y)}{p(x)p(y)}$$

Equivalently:
$$I(X; Y) = H(Y) - H(Y|X) = H(X) - H(X|Y)$$

### 1.2 The Information Bottleneck Principle

The Information Bottleneck seeks a representation T of X by solving:

$$\min_{p(t|x)} \left[ I(X; T) - \beta I(T; Y) \right]$$

Where:
- $I(X; T)$: Information that T preserves about X (complexity)
- $I(T; Y)$: Information that T preserves about Y (relevance)
- $\beta$: Trade-off parameter (Lagrange multiplier)

The optimal solution satisfies:

$$p(t|x) = \frac{p(t)}{Z(x, \beta)} \exp\left(-\beta D_{KL}[p(y|x) || p(y|t)]\right)$$

Where $Z(x, \beta)$ is a normalization factor and $D_{KL}$ is the Kullback-Leibler divergence.

In [ ]:
# Helper functions for information theory calculations

def entropy(p):
    """Calculate entropy of a probability distribution."""
    p = np.array(p)
    p = p[p > 0]  # Remove zero probabilities
    return -np.sum(p * np.log2(p))

def mutual_information(joint_prob):
    """Calculate mutual information from joint probability distribution."""
    joint_prob = np.array(joint_prob)
    px = np.sum(joint_prob, axis=1, keepdims=True)
    py = np.sum(joint_prob, axis=0, keepdims=True)
    
    # Compute mutual information
    mi = 0
    for i in range(joint_prob.shape[0]):
        for j in range(joint_prob.shape[1]):
            if joint_prob[i, j] > 0:
                mi += joint_prob[i, j] * np.log2(joint_prob[i, j] / (px[i, 0] * py[0, j]))
    return mi

def conditional_entropy(joint_prob):
    """Calculate conditional entropy H(Y|X) from joint probability."""
    joint_prob = np.array(joint_prob)
    px = np.sum(joint_prob, axis=1, keepdims=True)
    
    h_cond = 0
    for i in range(joint_prob.shape[0]):
        if px[i, 0] > 0:
            p_y_given_x = joint_prob[i, :] / px[i, 0]
            h_cond += px[i, 0] * entropy(p_y_given_x)
    return h_cond

print("Information theory helper functions defined.")

## 2. Synthetic Example: Binary Noisy Channel

Let's start with a simple example to understand how IB works. We'll consider a scenario where:
- **X**: A binary input signal (0 or 1) sent through a noisy channel
- **Y**: The true underlying state we want to infer (not directly observable)
- **T**: A compressed representation of X

### Scenario: Medical Diagnosis
Imagine:
- **Y**: Disease state (Healthy=0, Diseased=1)
- **X**: Multiple noisy test results
- **T**: Compressed diagnostic indicator

We want to compress the test results (X) into a simple indicator (T) that still predicts the disease state (Y) accurately.

In [ ]:
# Create a synthetic binary channel example
np.random.seed(42)

# Define the true state Y (disease state)
n_samples = 1000
Y = np.random.binomial(1, 0.3, n_samples)  # 30% disease prevalence

# Define X as noisy observations of Y
# When Y=0 (healthy): X tends to be low values
# When Y=1 (diseased): X tends to be high values
X = np.zeros(n_samples, dtype=int)

for i in range(n_samples):
    if Y[i] == 0:
        # Healthy: observations are 0, 1, 2 with different probabilities
        X[i] = np.random.choice([0, 1, 2], p=[0.6, 0.3, 0.1])
    else:
        # Diseased: observations are 1, 2, 3 with different probabilities
        X[i] = np.random.choice([1, 2, 3], p=[0.1, 0.3, 0.6])

# Compute empirical joint distribution p(x, y)
joint_counts = np.zeros((4, 2))  # 4 possible X values (0-3), 2 possible Y values (0-1)
for i in range(n_samples):
    joint_counts[X[i], Y[i]] += 1

joint_prob = joint_counts / n_samples

print("Joint Probability Distribution p(X, Y):")
print(pd.DataFrame(joint_prob, columns=['Y=0 (Healthy)', 'Y=1 (Diseased)'], 
                   index=['X=0', 'X=1', 'X=2', 'X=3']))
print(f"\nMutual Information I(X;Y) = {mutual_information(joint_prob):.3f} bits")

In [ ]:
# Visualize the joint distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap of joint probability
sns.heatmap(joint_prob, annot=True, fmt='.3f', cmap='YlOrRd', 
            xticklabels=['Healthy', 'Diseased'],
            yticklabels=['X=0', 'X=1', 'X=2', 'X=3'],
            ax=axes[0])
axes[0].set_title('Joint Distribution p(X, Y)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Observation X', fontsize=12)
axes[0].set_xlabel('True State Y', fontsize=12)

# Conditional distributions p(Y|X)
px = np.sum(joint_prob, axis=1, keepdims=True)
cond_prob = joint_prob / (px + 1e-10)

x_vals = np.arange(4)
width = 0.35
axes[1].bar(x_vals - width/2, cond_prob[:, 0], width, label='P(Healthy|X)', alpha=0.8)
axes[1].bar(x_vals + width/2, cond_prob[:, 1], width, label='P(Diseased|X)', alpha=0.8)
axes[1].set_xlabel('Observation X', fontsize=12)
axes[1].set_ylabel('Conditional Probability', fontsize=12)
axes[1].set_title('Conditional Distribution p(Y|X)', fontsize=14, fontweight='bold')
axes[1].set_xticks(x_vals)
axes[1].set_xticklabels(['X=0', 'X=1', 'X=2', 'X=3'])
axes[1].legend()
axes[1].set_ylim([0, 1])

plt.tight_layout()
plt.show()

print("\nKey Insight: X=0 strongly indicates healthy, X=3 strongly indicates diseased.")
print("X=1 and X=2 are more ambiguous and appear in both states.")

### Information Bottleneck: Finding Optimal Compression

Now, let's apply the IB principle to find a compressed representation T. We'll consider different compression schemes:

1. **No compression**: T = X (4 states)
2. **Binary compression**: Merge X into 2 clusters
3. **Maximum compression**: Single state (loses all information)

In [ ]:
# Implement different compression schemes

def compute_ib_quantities(joint_xy, mapping_xt):
    """
    Compute I(X;T) and I(T;Y) for a given mapping from X to T.
    
    Args:
        joint_xy: Joint distribution p(x,y)
        mapping_xt: Array where mapping_xt[x] = t (deterministic mapping)
    
    Returns:
        I_XT, I_TY: Mutual informations
    """
    n_t = len(np.unique(mapping_xt))
    n_x, n_y = joint_xy.shape
    
    # Compute joint distribution p(t, x)
    joint_tx = np.zeros((n_t, n_x))
    for x in range(n_x):
        t = mapping_xt[x]
        joint_tx[t, x] = np.sum(joint_xy[x, :])
    
    # Compute joint distribution p(t, y)
    joint_ty = np.zeros((n_t, n_y))
    for x in range(n_x):
        t = mapping_xt[x]
        joint_ty[t, :] += joint_xy[x, :]
    
    # Calculate mutual informations
    I_XT = mutual_information(joint_tx.T)  # Transpose to match (X, T) format
    I_TY = mutual_information(joint_ty)
    
    return I_XT, I_TY

# Test different compression schemes
compression_schemes = {
    'No compression (T=X)': np.array([0, 1, 2, 3]),
    'Binary: {0,1} vs {2,3}': np.array([0, 0, 1, 1]),
    'Binary: {0} vs {1,2,3}': np.array([0, 1, 1, 1]),
    'Binary: {0,1,2} vs {3}': np.array([0, 0, 0, 1]),
    'Maximum compression': np.array([0, 0, 0, 0])
}

results = []
for name, mapping in compression_schemes.items():
    I_XT, I_TY = compute_ib_quantities(joint_prob, mapping)
    results.append({
        'Scheme': name,
        'I(X;T)': I_XT,
        'I(T;Y)': I_TY,
        'Compression': 1 - I_XT / mutual_information(joint_prob)
    })

results_df = pd.DataFrame(results)
print("\nCompression Schemes Comparison:")
print(results_df.to_string(index=False))
print(f"\nReference: I(X;Y) = {mutual_information(joint_prob):.3f} bits")

In [ ]:
# Visualize the Information Bottleneck trade-off curve
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# Extract I(X;T) and I(T;Y) for plotting
I_XT_vals = results_df['I(X;T)'].values
I_TY_vals = results_df['I(T;Y)'].values

# Plot the trade-off curve
ax.scatter(I_XT_vals, I_TY_vals, s=200, c='red', alpha=0.6, edgecolors='black', linewidth=2)

# Add labels for each point
for i, name in enumerate(results_df['Scheme']):
    # Shorten labels for better visibility
    short_name = name.split(':')[0] if ':' in name else name
    ax.annotate(short_name, (I_XT_vals[i], I_TY_vals[i]), 
                xytext=(10, 10), textcoords='offset points',
                fontsize=9, bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.5))

# Draw a reference line for I(X;Y)
I_XY = mutual_information(joint_prob)
ax.axhline(y=I_XY, color='blue', linestyle='--', linewidth=2, label=f'I(X;Y) = {I_XY:.3f} bits')

ax.set_xlabel('I(X;T) - Complexity (bits)', fontsize=12, fontweight='bold')
ax.set_ylabel('I(T;Y) - Relevance (bits)', fontsize=12, fontweight='bold')
ax.set_title('Information Bottleneck Trade-off Curve', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== Interpretation ===")
print("• Points closer to upper-left preserve more relevant information with less complexity")
print("• The optimal compression depends on the trade-off parameter β")
print("• The blue line shows the maximum achievable I(T;Y) = I(X;Y)")

### Key Observations from Synthetic Example

1. **Trade-off**: As we compress X into T (reducing I(X;T)), we lose information about Y (reducing I(T;Y))
2. **Optimal compression**: The binary scheme {0,1} vs {2,3} provides a good balance
3. **Information bottleneck**: We can't preserve more information about Y than what X contains: I(T;Y) ≤ I(X;Y)

## 3. Practical Real-World Example: Document Clustering

Now let's apply the Information Bottleneck principle to a real-world problem: **document clustering**.

### Problem Setup
- **X**: Document representations (TF-IDF vectors)
- **Y**: Document topics/categories (latent)
- **T**: Cluster assignments (compressed representation)

We'll use a set of documents about different topics and see how IB-inspired clustering can group similar documents while preserving topic information.

In [ ]:
# Create a synthetic document dataset
documents = [
    # Sports documents
    "The football team won the championship game with an amazing goal",
    "Basketball players scored many points in the final match",
    "The tennis tournament featured incredible athletes competing",
    "Soccer fans celebrated the victory in the stadium",
    "The baseball game went into extra innings last night",
    
    # Technology documents
    "Machine learning algorithms improve computer vision systems",
    "Artificial intelligence transforms data analysis and prediction",
    "Neural networks process information like human brains",
    "Deep learning models achieve breakthrough performance",
    "Software engineers develop innovative applications daily",
    
    # Medicine documents
    "Doctors diagnose patients using advanced medical imaging",
    "New treatment shows promising results in clinical trials",
    "Medical research advances understanding of diseases",
    "Hospital staff provide care to patients every day",
    "Pharmaceutical companies develop new therapeutic drugs",
    
    # Finance documents
    "Stock market prices fluctuate based on economic indicators",
    "Investors analyze financial reports for trading decisions",
    "Banking sector shows strong quarterly revenue growth",
    "Cryptocurrency trading reaches new market highs",
    "Economic policy impacts interest rates and inflation"
]

# True labels (topics)
true_labels = (
    [0] * 5 +  # Sports
    [1] * 5 +  # Technology
    [2] * 5 +  # Medicine
    [3] * 5    # Finance
)

topic_names = ['Sports', 'Technology', 'Medicine', 'Finance']

print(f"Dataset: {len(documents)} documents across {len(topic_names)} topics")
print(f"Topics: {', '.join(topic_names)}")

In [ ]:
# Convert documents to TF-IDF features
vectorizer = TfidfVectorizer(max_features=50, stop_words='english')
X_tfidf = vectorizer.fit_transform(documents).toarray()

print(f"Feature matrix shape: {X_tfidf.shape}")
print(f"\nTop features: {', '.join(vectorizer.get_feature_names_out()[:10])}")

# Apply K-means clustering (IB-inspired approach)
# Different numbers of clusters (different compression levels)
n_clusters_range = [2, 3, 4, 5, 6]
clustering_results = {}

for n_clusters in n_clusters_range:
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X_tfidf)
    clustering_results[n_clusters] = cluster_labels

print(f"\nClustered documents with {len(n_clusters_range)} different compression levels")

In [ ]:
# Compute information-theoretic quantities for each clustering
def compute_clustering_info(cluster_labels, true_labels):
    """
    Compute I(X;T) approximation and I(T;Y) for clustering.
    """
    n_samples = len(cluster_labels)
    n_clusters = len(np.unique(cluster_labels))
    n_classes = len(np.unique(true_labels))
    
    # Build confusion matrix (joint distribution)
    confusion = np.zeros((n_clusters, n_classes))
    for i in range(n_samples):
        confusion[cluster_labels[i], true_labels[i]] += 1
    
    joint_prob = confusion / n_samples
    
    # Compute I(T;Y)
    I_TY = mutual_information(joint_prob)
    
    # Approximate I(X;T) using cluster entropy
    # H(T) represents how much information is preserved
    p_t = np.sum(joint_prob, axis=1)
    H_T = entropy(p_t)
    
    return H_T, I_TY, confusion

info_results = []
for n_clusters, cluster_labels in clustering_results.items():
    H_T, I_TY, confusion = compute_clustering_info(cluster_labels, true_labels)
    info_results.append({
        'n_clusters': n_clusters,
        'H(T)': H_T,
        'I(T;Y)': I_TY
    })

info_df = pd.DataFrame(info_results)
print("\nClustering Information Analysis:")
print(info_df.to_string(index=False))

In [ ]:
# Visualize the information bottleneck trade-off for document clustering
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left plot: Trade-off curve
ax = axes[0]
ax.plot(info_df['H(T)'], info_df['I(T;Y)'], 'o-', markersize=10, linewidth=2, color='darkblue')

for i, row in info_df.iterrows():
    ax.annotate(f"{row['n_clusters']} clusters", 
                (row['H(T)'], row['I(T;Y)']),
                xytext=(10, -10), textcoords='offset points',
                fontsize=9, bbox=dict(boxstyle='round,pad=0.3', facecolor='lightblue', alpha=0.7))

ax.set_xlabel('H(T) - Complexity (bits)', fontsize=12, fontweight='bold')
ax.set_ylabel('I(T;Y) - Information about Topics (bits)', fontsize=12, fontweight='bold')
ax.set_title('Document Clustering: IB Trade-off', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

# Right plot: Confusion matrix for optimal clustering (4 clusters)
ax = axes[1]
_, _, confusion_4 = compute_clustering_info(clustering_results[4], true_labels)
sns.heatmap(confusion_4, annot=True, fmt='.0f', cmap='Blues', 
            xticklabels=topic_names,
            yticklabels=[f'Cluster {i}' for i in range(4)],
            ax=ax)
ax.set_title('Confusion Matrix (4 Clusters)', fontsize=14, fontweight='bold')
ax.set_xlabel('True Topic', fontsize=12)
ax.set_ylabel('Assigned Cluster', fontsize=12)

plt.tight_layout()
plt.show()

print("\n=== Analysis ===")
print("• As we increase the number of clusters, H(T) increases (more complex representation)")
print("• More clusters also capture more information about topics I(T;Y)")
print("• The optimal number of clusters balances complexity and topic preservation")
print("• With 4 clusters (matching 4 topics), we achieve good topic separation")

In [ ]:
# Show example documents from each cluster
print("\n=== Example Documents from Each Cluster (4-cluster solution) ===")
print()

cluster_labels_4 = clustering_results[4]

for cluster_id in range(4):
    print(f"Cluster {cluster_id}:")
    docs_in_cluster = [i for i, label in enumerate(cluster_labels_4) if label == cluster_id]
    
    # Show first 2 documents and their true topics
    for doc_idx in docs_in_cluster[:2]:
        true_topic = topic_names[true_labels[doc_idx]]
        print(f"  [{true_topic}] {documents[doc_idx][:60]}...")
    print()

## 4. Practical Insights and Applications

### Key Takeaways

1. **Compression-Prediction Trade-off**: The IB principle formalizes the fundamental trade-off between:
   - Compressing data (reducing I(X;T))
   - Maintaining predictive power (maximizing I(T;Y))

2. **Optimal Representations**: For any compression level β, there exists an optimal representation T that maximizes relevant information while minimizing irrelevant details

3. **Inference from Partial Observations**: 
   - Even when Y is not directly observable, we can learn about it through X
   - The bottleneck T extracts only the features of X that are relevant for predicting Y
   - Irrelevant variations in X are discarded

### Real-World Applications

1. **Deep Learning**: Understanding how neural networks learn hierarchical representations
2. **Feature Selection**: Choosing features that maximize predictive power while minimizing redundancy
3. **Clustering**: Grouping data points based on shared relevant information
4. **Dimensionality Reduction**: Compressing high-dimensional data while preserving task-relevant structure
5. **Privacy**: Creating data representations that preserve utility while discarding sensitive information

### Connection to Machine Learning

The Information Bottleneck provides a theoretical framework for understanding:
- **Generalization**: Why models that compress input representations often generalize better
- **Regularization**: The parameter β acts like a regularization parameter
- **Feature Learning**: How to automatically discover relevant features from raw data

## 5. Summary and Conclusions

### What We've Learned

1. **Mathematical Foundation**: The IB principle provides a rigorous information-theoretic framework for data compression

2. **Synthetic Example**: We demonstrated how IB works with a simple medical diagnosis scenario where:
   - Noisy test results (X) must be compressed
   - While preserving information about disease state (Y)
   - Different compression schemes show different trade-offs

3. **Real-World Application**: Document clustering showed how IB principles apply to practical problems:
   - Varying cluster numbers corresponds to different compression levels
   - Optimal clustering balances complexity and topic preservation
   - IB provides a principled way to choose the right compression level

### The Power of Information Bottleneck

The IB principle is powerful because it:
- Provides a **unified framework** for compression and prediction
- Offers **theoretical guarantees** about optimal representations
- Applies to diverse problems from clustering to deep learning
- Connects information theory, statistics, and machine learning

### Further Reading

- Tishby, N., Pereira, F. C., & Bialek, W. (1999). "The information bottleneck method."
- Tishby, N., & Zaslavsky, N. (2015). "Deep learning and the information bottleneck principle."
- Achille, A., & Soatto, S. (2018). "Emergence of invariance and disentanglement in deep representations."

---

**Thank you for working through this tutorial! The Information Bottleneck is a beautiful example of how information theory provides insights into learning and compression.**